# Listings data preparation 
In this notebook we prepare Airbnb listings dataset for predictive modeling. We aim to identify the variables that may explain listing prices in Rome and clean the dataset. 
Our target variable selected for this stage is the lsiting nightly price.

In [114]:
#setup
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: /Users/santiagotawata/Desktop/airbnb-rome-analysis


## 1. Load dataset

In [115]:
listings = pd.read_csv(
    PROJECT_ROOT / "data" / "listings.csv"
)
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2737,https://www.airbnb.com/rooms/2737,20250914152919,2025-09-15,city scrape,"Elif's room in cozy, clean flat.",10 min by bus you can get to Piazza Venezia or...,It used to be an industrial area until late 80...,https://a0.muscache.com/pictures/41225252/e955...,3047,...,5.00,4.40,4.40,NaN,f,6,0,6,0,0.04
1,11834,https://www.airbnb.com/rooms/11834,20250914152919,2025-09-15,city scrape,"Charming Boschetto Studio, Rome",Fantastic apartment in the Monti district. The...,"""Monti"" with its narrow cobblestone alleys, cr...",https://a0.muscache.com/pictures/miso/Hosting-...,44552,...,4.96,4.99,4.81,IT058091C29VJSIZQZ,f,1,1,0,0,1.62
2,12398,https://www.airbnb.com/rooms/12398,20250914152919,2025-09-15,city scrape,Casa Donatello - Home far from Home,Casa Donatello is a newly renovated two-bedroo...,You are at 15 minutes walking distance from hi...,https://a0.muscache.com/pictures/miso/Hosting-...,11756,...,5.00,4.89,4.83,it058091c2kv6epw8f,f,1,1,0,0,0.47
3,19965,https://www.airbnb.com/rooms/19965,20250914152919,2025-09-15,city scrape,S. Peter's Square 5 Min WALK bright and quite ...,AT ONLY 5 MINUTES WALK to S.Peter's Basilica S...,Prati is a famous neighbourhood (rione of Rome...,https://a0.muscache.com/pictures/hosting/Hosti...,75450,...,4.90,4.90,4.54,IT058091C20YD35BX2,t,3,3,0,0,1.07
4,19967,https://www.airbnb.com/rooms/19967,20250914152919,2025-09-15,city scrape,*In front Vatican Museums 2 bedrooms quite bri...,"IN FRONT of the Vatican Museums entrance, at O...",Prati is a famous neighbourhood (rione of Rome...,https://a0.muscache.com/pictures/hosting/Hosti...,75450,...,4.80,4.85,4.28,IT058091C20YD35BX2,t,3,3,0,0,0.32


## 2. Exploration

In [116]:
listings.shape

(37652, 79)

In [117]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37652 entries, 0 to 37651
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            37652 non-null  int64  
 1   listing_url                                   37652 non-null  object 
 2   scrape_id                                     37652 non-null  int64  
 3   last_scraped                                  37652 non-null  object 
 4   source                                        37652 non-null  object 
 5   name                                          37652 non-null  object 
 6   description                                   36703 non-null  object 
 7   neighborhood_overview                         17721 non-null  object 
 8   picture_url                                   37652 non-null  object 
 9   host_id                                       37652 non-null 

In [118]:
listings.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

## 4. Dataset cleaning
In the dataset the `price` variable is stored as text and contains currency symbols.
To use it in our model, it must be converted into a numeric format. 

In [119]:
listings["price"].head()

0     $57.00
1    $110.00
2    $124.00
3    $162.00
4    $150.00
Name: price, dtype: object

In [120]:
listings["price"].isna().sum()

np.int64(4088)

In [121]:
listings["price"] = (
    listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [122]:
# to verify
listings["price"].head()

0     57.0
1    110.0
2    124.0
3    162.0
4    150.0
Name: price, dtype: float64

Feature selection was guided by the adquire knowledge from Airbnb official webpage about marketplace, prioritazing location variables, accomodation characteristic, host reputation, and guest satisfaction. 
 ## 5. Selection of features
 Having in consideration the complete list of features, we decided to delete the following ones: 
 #### 5.1. Identifiers
 these variables identify the records but they do not contain  information that explains the prices of each residency. Including them would just add noise to the model. 
 #### 5.2. Textual description 
 This group of variables might contain useful information but we need NLP for their analysis. Therefore, we will also exclude them. 
 #### 5.3. Image and media
 Again, is not numerical useful information, it goes beyond our lecture. 
 #### 5.4. Administrative fields
 Variables that describe how the information was retrieved does not affect the listing price.
 #### 5.5. Redundant variables
In this group there are variable that have very similar information between them, if we were to include them we would only create redundance in our models. 
#### 5.6. Reviews
For the first part of our work we will exclude them, but they might be more useful when we apply feature engineering. 

In [123]:
candidate_features = [
    "price",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "host_is_superhost",
    "host_response_rate",
    "host_acceptance_rate",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "number_of_reviews",
    "reviews_per_month",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "amenities",    
    "first_review",      
    "last_review",  
    "last_scraped"
]

# Candidate Features Data Dictionary

| Field | Type | Description |
|---|---|---|
| `price` | currency | Daily price in local currency. Note: the `$` sign is a technical artifact of the export and should be ignored. |
| `neighbourhood_cleansed` | text | The neighbourhood as geocoded using the listing's latitude and longitude against neighborhoods defined by open/public digital shapefiles. |
| `latitude` | numeric | Latitude of the listing, using the WGS84 projection. |
| `longitude` | numeric | Longitude of the listing, using the WGS84 projection. |
| `property_type` | text | Self-selected property type, as entered by the host (e.g. Hotels and Bed and Breakfasts are described as such by their hosts). |
| `room_type` | text | One of `Entire home/apt`, `Private room`, `Shared room`, or `Hotel`. Indicates how much of the space the guest has exclusive access to. |
| `accommodates` | integer | The maximum guest capacity of the listing. |
| `bedrooms` | integer | The number of bedrooms in the listing. |
| `beds` | integer | The number of beds in the listing. |
| `bathrooms` | numeric | The number of bathrooms in the listing. |
| `host_is_superhost` | boolean | `t` = true, `f` = false. Whether the host holds Superhost status. |
| `host_response_rate` | percentage | The rate at which the host responds to new messages/inquiries. |
| `host_acceptance_rate` | percentage | The rate at which the host accepts booking requests. |
| `review_scores_rating` | numeric | The overall review rating score for the listing, as given by past guests. |
| `review_scores_cleanliness` | numeric | The review sub-score for cleanliness, as given by past guests. |
| `review_scores_location` | numeric | The review sub-score for location, as given by past guests. |
| `review_scores_value` | numeric | The review sub-score for value, as given by past guests. |
| `number_of_reviews` | integer | The total number of reviews the listing has received. |
| `reviews_per_month` | numeric | The average number of reviews per month over the lifetime of the listing (calculated field). |
| `minimum_nights` | integer | Minimum number of nights required for a stay (calendar-specific rules may override this). |
| `maximum_nights` | integer | Maximum number of nights allowed for a stay (calendar-specific rules may override this). |
| `instant_bookable` | boolean | `t` = true, `f` = false. Whether a guest can book the listing automatically, without the host needing to approve the request — often an indicator of a commercial listing. |

In [124]:
selected = listings[candidate_features]

selected.head()

,price,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,bathrooms,...,review_scores_value,number_of_reviews,reviews_per_month,minimum_nights,maximum_nights,instant_bookable,amenities,first_review,last_review,last_scraped
0,57.0,VIII Appia Antica,41.871360,12.482150,Private room,Private room,1,1.0,1.0,1.5,...,4.40,5,0.04,31,1125,f,"[""Hangers"", ""Heating"", ""Free parking on premis...",2014-12-26,2015-05-08,2025-09-15
1,110.0,I Centro Storico,41.895447,12.491181,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,...,4.81,284,1.62,2,28,f,"[""Hangers"", ""Pack \u2019n play/Travel crib"", ""...",2011-05-01,2025-07-05,2025-09-15
2,124.0,II Parioli/Nomentano,41.925820,12.469280,Entire rental unit,Entire home/apt,6,2.0,3.0,1.0,...,4.83,85,0.47,3,365,f,"[""Hangers"", ""Heating"", ""50 inch HDTV with stan...",2010-10-18,2025-08-01,2025-09-15
3,162.0,I Centro Storico,41.908230,12.452930,Entire condo,Entire home/apt,5,2.0,3.0,1.0,...,4.54,178,1.07,3,365,t,"[""Hangers"", ""Pack \u2019n play/Travel crib"", ""...",2012-01-10,2025-08-05,2025-09-15
4,150.0,I Centro Storico,41.908283,12.452617,Entire rental unit,Entire home/apt,5,2.0,3.0,1.0,...,4.28,46,0.32,3,365,t,"[""Hangers"", ""Ceiling fan"", ""Carbon monoxide al...",2013-10-17,2024-07-19,2025-09-15


In [125]:
selected = selected.dropna(subset=["price"])

## 6. Missing values analysis
This is an important step to comprehend the information that our model carries and therefore after being able to realise a proper selection of variables. 

In [126]:
missing_table = pd.DataFrame({
    "missing_count": selected.isna().sum(),
    "missing_pct": selected.isna().mean() * 100
})

missing_table = (
    missing_table
    .sort_values("missing_pct", ascending=False)
)

missing_table

,missing_count,missing_pct
host_response_rate,4737,14.113336
review_scores_value,4297,12.802407
review_scores_location,4297,12.802407
review_scores_cleanliness,4296,12.799428
reviews_per_month,4295,12.796449
review_scores_rating,4295,12.796449
first_review,4295,12.796449
last_review,4295,12.796449
host_acceptance_rate,2432,7.245859
host_is_superhost,2034,6.060064


In [127]:
missing_table[missing_table["missing_count"] > 0]

,missing_count,missing_pct
host_response_rate,4737,14.113336
review_scores_value,4297,12.802407
review_scores_location,4297,12.802407
review_scores_cleanliness,4296,12.799428
reviews_per_month,4295,12.796449
review_scores_rating,4295,12.796449
first_review,4295,12.796449
last_review,4295,12.796449
host_acceptance_rate,2432,7.245859
host_is_superhost,2034,6.060064


## 7. Correction of percentage variables
`host_response_rate`
`host_acceptance_rate`
are stored as percentages and must be converted into numerical values.

In [128]:
selected["host_response_rate"].head()

0      0%
1    100%
2     NaN
3    100%
4    100%
Name: host_response_rate, dtype: object

In [129]:
selected["host_acceptance_rate"].head()

0      0%
1     95%
2    100%
3     99%
4     99%
Name: host_acceptance_rate, dtype: object

In [130]:
selected["host_response_rate"] = (
    selected["host_response_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

In [131]:
selected["host_acceptance_rate"] = (
    selected["host_acceptance_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

In [132]:
selected["host_response_rate"] = pd.to_numeric(
    selected["host_response_rate"],
    errors="coerce"
)

selected["host_acceptance_rate"] = pd.to_numeric(
    selected["host_acceptance_rate"],
    errors="coerce"
)

In [133]:
selected[["host_response_rate", "host_acceptance_rate"]].describe()

,host_response_rate,host_acceptance_rate
count,28827.000000,31132.000000
mean,96.758005,93.019466
std,13.471273,19.524055
min,0.000000,0.000000
25%,100.000000,98.000000
50%,100.000000,100.000000
75%,100.000000,100.000000
max,100.000000,100.000000


## 8. Imputation strategy 
For the following numerical variables we will impute the median:

- bedrooms
- beds
- bathrooms
- host_response_rate
- host_acceptance_rate
- review_scores_rating
- review_scores_cleanliness
- review_scores_location
- review_scores_value
- reviews_per_month

Median imputation is robust to extreme values and preserves the  distribution better than the mean.

In [134]:
numerical_columns = [
    "bedrooms",
    "beds",
    "bathrooms",
    "host_response_rate",
    "host_acceptance_rate",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month"
]

for col in numerical_columns:
    selected[col] = selected[col].fillna(
        selected[col].median()
    )

For the categorical variables, we will simply impute the value "Unkwown"

- property_type
- room_type
- host_is_superhost
- instant_bookable

This allows us to still preserve the information about missingness

In [135]:
categorical_columns = [
    "property_type",
    "room_type",
    "host_is_superhost",
    "instant_bookable"
]

for col in categorical_columns:
    selected[col] = selected[col].fillna("Unknown")

In [136]:
# verification
selected.isna().sum()

price                           0
neighbourhood_cleansed          0
latitude                        0
longitude                       0
property_type                   0
room_type                       0
accommodates                    0
bedrooms                        0
beds                            0
bathrooms                       0
host_is_superhost               0
host_response_rate              0
host_acceptance_rate            0
review_scores_rating            0
review_scores_cleanliness       0
review_scores_location          0
review_scores_value             0
number_of_reviews               0
reviews_per_month               0
minimum_nights                  0
maximum_nights                  0
instant_bookable                0
amenities                       0
first_review                 4295
last_review                  4295
last_scraped                    0
dtype: int64

## 9. Properties Features Engineering
To better represent the relationship between accommodation capacity and available resources, we create ratio-based features that capture bed availability and bathroom density per guest.

In [137]:
# Beds per guest

selected["beds_per_guest"] = np.where(
    selected["accommodates"] > 0,
    selected["beds"] / selected["accommodates"],
    np.nan
)

# Bathrooms per guest

selected["bathrooms_per_guest"] = np.where(
    selected["accommodates"] > 0,
    selected["bathrooms"] / selected["accommodates"],
    np.nan
)

In [138]:
selected[
    [
        "accommodates",
        "beds",
        "bathrooms",
        "beds_per_guest",
        "bathrooms_per_guest"
    ]
].head()

,accommodates,beds,bathrooms,beds_per_guest,bathrooms_per_guest
0,1,1.0,1.5,1.0,1.500000
1,2,1.0,1.0,0.5,0.500000
2,6,3.0,1.0,0.5,0.166667
3,5,3.0,1.0,0.6,0.200000
4,5,3.0,1.0,0.6,0.200000


In [139]:
selected[
    [
        "beds_per_guest",
        "bathrooms_per_guest"
    ]
].isna().sum()

beds_per_guest         0
bathrooms_per_guest    0
dtype: int64

In [140]:
selected[
    [
        "beds_per_guest",
        "bathrooms_per_guest"
    ]
].describe()

,beds_per_guest,bathrooms_per_guest
count,33564.000000,33564.000000
mean,0.586774,0.385350
std,0.273847,0.196680
min,0.000000,0.000000
25%,0.500000,0.250000
50%,0.500000,0.333333
75%,0.666667,0.500000
max,10.000000,6.000000


In [141]:
selected["amenities"] = listings["amenities"]

In [142]:
listings["amenities"].head(3)

0    ["Hangers", "Heating", "Free parking on premis...
1    ["Hangers", "Pack \u2019n play/Travel crib", "...
2    ["Hangers", "Heating", "50 inch HDTV with stan...
Name: amenities, dtype: object

In [143]:
selected.columns.tolist()

['price',
 'neighbourhood_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bedrooms',
 'beds',
 'bathrooms',
 'host_is_superhost',
 'host_response_rate',
 'host_acceptance_rate',
 'review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_location',
 'review_scores_value',
 'number_of_reviews',
 'reviews_per_month',
 'minimum_nights',
 'maximum_nights',
 'instant_bookable',
 'amenities',
 'first_review',
 'last_review',
 'last_scraped',
 'beds_per_guest',
 'bathrooms_per_guest']

In [144]:
selected["amenities"].sample(5)

15493    ["Hangers", "Ceiling fan", "Carbon monoxide al...
3511     ["Hangers", "Heating", "Carbon monoxide alarm"...
36326    ["Air conditioning", "Dedicated workspace", "W...
20967    ["Hangers", "Free washer", "Pack \u2019n play/...
20774    ["Hangers", "Ceiling fan", "Carbon monoxide al...
Name: amenities, dtype: object

## 10. Amenities Feature Engineering
The amenities column contains a list of services and facilities offered by each property.

Instead of using the raw text, we create binary indicators for the most relevant amenities and an aggregated amenities count variable.

In [145]:
selected["amenities_count"] = (
    selected["amenities"]
    .str.count(",")
    .fillna(0)
    + 1
)

In [146]:
selected["amenities_count"].describe()

count    33564.000000
mean        33.971547
std         14.675557
min          1.000000
25%         24.000000
50%         35.000000
75%         44.000000
max        126.000000
Name: amenities_count, dtype: float64

In [147]:
selected["has_wifi"] = (
    selected["amenities"]
    .str.contains("Wifi", case=False, na=False)
    .astype(int)
)

selected["has_air_conditioning"] = (
    selected["amenities"]
    .str.contains("Air conditioning", case=False, na=False)
    .astype(int)
)

selected["has_kitchen"] = (
    selected["amenities"]
    .str.contains("Kitchen", case=False, na=False)
    .astype(int)
)

selected["has_washer"] = (
    selected["amenities"]
    .str.contains("Washer", case=False, na=False)
    .astype(int)
)

selected["has_dryer"] = (
    selected["amenities"]
    .str.contains("Dryer", case=False, na=False)
    .astype(int)
)

selected["has_parking"] = (
    selected["amenities"]
    .str.contains("parking", case=False, na=False)
    .astype(int)
)

selected["has_elevator"] = (
    selected["amenities"]
    .str.contains("Elevator", case=False, na=False)
    .astype(int)
)

selected["has_tv"] = (
    selected["amenities"]
    .str.contains("TV", case=False, na=False)
    .astype(int)
)

selected["has_workspace"] = (
    selected["amenities"]
    .str.contains("workspace", case=False, na=False)
    .astype(int)
)

In [148]:
selected[
    [
        "has_wifi",
        "has_air_conditioning",
        "has_kitchen",
        "has_washer",
        "has_dryer",
        "has_parking",
        "has_elevator",
        "has_tv",
        "has_workspace"
    ]
].mean().sort_values(ascending=False)

has_wifi                0.982809
has_tv                  0.916965
has_dryer               0.892236
has_kitchen             0.857705
has_air_conditioning    0.770498
has_washer              0.681176
has_parking             0.564176
has_workspace           0.521005
has_elevator            0.434573
dtype: float64

In [149]:
selected = selected.drop(columns=["amenities"])

In [150]:
selected.columns.tolist()

['price',
 'neighbourhood_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bedrooms',
 'beds',
 'bathrooms',
 'host_is_superhost',
 'host_response_rate',
 'host_acceptance_rate',
 'review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_location',
 'review_scores_value',
 'number_of_reviews',
 'reviews_per_month',
 'minimum_nights',
 'maximum_nights',
 'instant_bookable',
 'first_review',
 'last_review',
 'last_scraped',
 'beds_per_guest',
 'bathrooms_per_guest',
 'amenities_count',
 'has_wifi',
 'has_air_conditioning',
 'has_kitchen',
 'has_washer',
 'has_dryer',
 'has_parking',
 'has_elevator',
 'has_tv',
 'has_workspace']

In [151]:
selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33564 entries, 0 to 37651
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   price                      33564 non-null  float64
 1   neighbourhood_cleansed     33564 non-null  object 
 2   latitude                   33564 non-null  float64
 3   longitude                  33564 non-null  float64
 4   property_type              33564 non-null  object 
 5   room_type                  33564 non-null  object 
 6   accommodates               33564 non-null  int64  
 7   bedrooms                   33564 non-null  float64
 8   beds                       33564 non-null  float64
 9   bathrooms                  33564 non-null  float64
 10  host_is_superhost          33564 non-null  object 
 11  host_response_rate         33564 non-null  float64
 12  host_acceptance_rate       33564 non-null  float64
 13  review_scores_rating       33564 non-null  float64


## 11. Host Features Proceessing 
Host-related characteristics are often strong predictors of listing performance and pricing.

We verify and standardize host quality indicators before modeling.

In [152]:
selected["host_is_superhost"].value_counts(dropna=False)

host_is_superhost
f          18493
t          13037
Unknown     2034
Name: count, dtype: int64

In [153]:
#superhost encoding 
selected["host_is_superhost"] = (
    selected["host_is_superhost"]
    .map({
        "t": 1,
        "f": 0,
        "Unknown": 0
    })
)

In [154]:
selected["host_is_superhost"].value_counts(dropna=False)

host_is_superhost
0    20527
1    13037
Name: count, dtype: int64

In [155]:
selected["instant_bookable"].value_counts(dropna=False)

instant_bookable
t    20732
f    12832
Name: count, dtype: int64

In [156]:
selected["room_type"].value_counts(dropna=False)

room_type
Entire home/apt    26030
Private room        7253
Hotel room           205
Shared room           76
Name: count, dtype: int64

In [157]:
selected["property_type"].nunique()

61

In [158]:
selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33564 entries, 0 to 37651
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   price                      33564 non-null  float64
 1   neighbourhood_cleansed     33564 non-null  object 
 2   latitude                   33564 non-null  float64
 3   longitude                  33564 non-null  float64
 4   property_type              33564 non-null  object 
 5   room_type                  33564 non-null  object 
 6   accommodates               33564 non-null  int64  
 7   bedrooms                   33564 non-null  float64
 8   beds                       33564 non-null  float64
 9   bathrooms                  33564 non-null  float64
 10  host_is_superhost          33564 non-null  int64  
 11  host_response_rate         33564 non-null  float64
 12  host_acceptance_rate       33564 non-null  float64
 13  review_scores_rating       33564 non-null  float64


## 12. Review Features Engineering

Review-based features capture two distinct signals that may influence listing price:

- **Review quality**: how guests perceive the listing (cleanliness, location, value)
- **Review activity**: how frequently and recently reviews are written, as a proxy for demand

Listings with no reviews (~14% of the dataset) are handled separately to avoid introducing bias.

We create the following features:

| Feature | Description |
|---|---|
| `review_quality_index` | Mean of rating, cleanliness, location, value scores |
| `listing_age_days` | Days between first and last review (proxy for listing age) |
| `review_recency_days` | Days between the last review and the scraped date (lower = more active listing) |
| `review_intensity` | frequency of reviews in the active period(demand proxy) |

In [159]:
# Convert date columns to datetime
selected['last_scraped']  = pd.to_datetime(selected['last_scraped'],  errors='coerce')
selected['first_review']  = pd.to_datetime(selected['first_review'],  errors='coerce')
selected['last_review']   = pd.to_datetime(selected['last_review'],   errors='coerce')

# Fill last_scraped if missing (reference date of the scrape)
selected['last_scraped'] = selected['last_scraped'].fillna(pd.Timestamp('2025-09-15'))

print('Null counts in date columns:')
print(selected[['first_review', 'last_review', 'last_scraped']].isnull().sum())

Null counts in date columns:
first_review    4295
last_review     4295
last_scraped       0
dtype: int64


Fill last_scraped if missing (reference date of the scrape).

In [160]:
selected['last_scraped'] = selected['last_scraped'].fillna(pd.Timestamp('2025-09-15'))

Verify that there are no nulls in the dataset

In [161]:


print('Null counts in date columns:')
print(selected[['first_review', 'last_review', 'last_scraped']].isnull().sum())

Null counts in date columns:
first_review    4295
last_review     4295
last_scraped       0
dtype: int64



We combine four review score dimensions into a single composite index.
These scores are already imputed (from previous steps), so no nulls remain. Basically, the method `mean(axis=1)` sums up the 4 parameters and divided the sum for the number of parameters. Then it use the method `described()` to show the principal measures in statistics.

In [162]:
review_score_cols = [
    'review_scores_rating',
    'review_scores_cleanliness',
    'review_scores_location',
    'review_scores_value'
]

selected['review_quality_index'] = selected[review_score_cols].mean(axis=1)

print('review_quality_index — sample stats:')
print(selected['review_quality_index'].describe().round(3))

review_quality_index — sample stats:
count    33564.000
mean         4.768
std          0.250
min          1.000
25%          4.712
50%          4.840
75%          4.892
max          5.000
Name: review_quality_index, dtype: float64


First we select the recently data as the reference scrape date.

In [163]:
scrape_date = selected['last_scraped'].max()  # reference date: 2025-09-15

`listing_age_days`

In [164]:
# Listing age in days (first to last review)
selected['listing_age_days'] = (
    (selected['last_review'] - selected['first_review'])
    .dt.days
    .fillna(0)
    .clip(lower=0)
)

`review_recency_days`

In [165]:

# Review recency: days since last review
selected['review_recency_days'] = (
    (scrape_date - selected['last_review'])
    .dt.days
)

`review_recency_days`

In [166]:
# No reviews → assign max recency (least active)
max_recency = selected['review_recency_days'].max()
selected['review_recency_days'] = selected['review_recency_days'].fillna(max_recency)

`review_intensity`

In [167]:
# Review intensity: reviews per day of listing age
selected['review_intensity'] = np.where(
    selected['listing_age_days'] > 0,
    selected['number_of_reviews'] / selected['listing_age_days'],
    0
)

Display of the measures of review activities

In [168]:
print('New temporal features:')
print(selected[['listing_age_days','review_recency_days','review_intensity']].describe().round(3))

New temporal features:
       listing_age_days  review_recency_days  review_intensity
count         33564.000            33564.000         33564.000
mean            961.235              731.509             0.066
std            1180.469             1549.994             0.088
min               0.000                0.000             0.000
25%              72.000               14.000             0.011
50%             464.000               50.000             0.048
75%            1360.000              316.000             0.099
max            5571.000             4690.000             2.000


Drop intermediate date columns

The raw date columns are not needed for modeling — we drop them after feature extraction.

In [169]:
selected.drop(columns=['first_review', 'last_review', 'last_scraped'], inplace=True, errors='ignore')

print('Review features added. Current shape:', selected.shape)
print('New columns:', ['review_quality_index','listing_age_days','review_recency_days','review_intensity'])

Review features added. Current shape: (33564, 38)
New columns: ['review_quality_index', 'listing_age_days', 'review_recency_days', 'review_intensity']


In [170]:
# Final check — no nulls in new features
new_feat = ['review_quality_index','listing_age_days','review_recency_days','review_intensity']
print('Null counts in review features:')
print(selected[new_feat].isnull().sum())

Null counts in review features:
review_quality_index    0
listing_age_days        0
review_recency_days     0
review_intensity        0
dtype: int64


In [171]:
selected.head()

,price,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,bathrooms,...,has_washer,has_dryer,has_parking,has_elevator,has_tv,has_workspace,review_quality_index,listing_age_days,review_recency_days,review_intensity
0,57.0,VIII Appia Antica,41.871360,12.482150,Private room,Private room,1,1.0,1.0,1.5,...,1,1,1,1,0,1,4.5500,133.0,3783.0,0.037594
1,110.0,I Centro Storico,41.895447,12.491181,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,...,0,1,1,0,1,0,4.8975,5179.0,72.0,0.054837
2,124.0,II Parioli/Nomentano,41.925820,12.469280,Entire rental unit,Entire home/apt,6,2.0,3.0,1.0,...,1,1,1,1,1,1,4.8800,5401.0,45.0,0.015738
3,162.0,I Centro Storico,41.908230,12.452930,Entire condo,Entire home/apt,5,2.0,3.0,1.0,...,1,1,1,1,1,1,4.6175,4956.0,41.0,0.035916
4,150.0,I Centro Storico,41.908283,12.452617,Entire rental unit,Entire home/apt,5,2.0,3.0,1.0,...,1,1,1,1,1,1,4.3575,3928.0,423.0,0.011711
